# 13 — Audit trail + operator identity + writable history (phase 5)

Every answer the copilot delivers is recorded — what was recommended, from which manual pages, to whom, when, at what cost, and whether the safety gate touched it — in a stdlib SQLite db (`factory_floor/audit.py`, `journal_mode=WAL`). Operators sign in against a committed `operators.csv` (PBKDF2-hashed PINs). Their actual resolution steps go back into the machine's history; the static `maintenance_history.csv` stays read-only and the UI unions the two.

This notebook runs the whole flow against a **temporary** db so it leaves no state.

In [1]:
import sys, tempfile
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import os
_tmp = Path(tempfile.mkdtemp())
os.environ['FACTORY_FLOOR_AUDIT_DB_PATH'] = str(_tmp / 'audit.sqlite3')
os.environ['FACTORY_FLOOR_CMMS_OUTBOX_PATH'] = str(_tmp / 'cmms_outbox.jsonl')

from factory_floor.config import get_settings
get_settings.cache_clear()
print('audit db  ->', get_settings().audit_db_path)

audit db  -> /var/folders/gx/gyggrc894m1dfwn7kzs5zq7m0000gn/T/tmp1514elpo/audit.sqlite3


## Operator sign-in

In [2]:
from factory_floor.identity import authenticate, list_operators

print('roster:', [(o.operator_id, o.name, o.role) for o in list_operators()])
operator = authenticate('OP-1001', '1234')
assert operator is not None and operator.name == 'Ana Costa'
assert authenticate('OP-1001', 'wrong') is None
operator

roster: [('OP-1001', 'Ana Costa', 'technician'), ('OP-1002', 'Bruno Silva', 'technician'), ('OP-2001', 'Carla Dias', 'supervisor')]


Operator(operator_id='OP-1001', name='Ana Costa', role='technician', tenant_id='default')

## A diagnostic turn, recorded

In [3]:
from factory_floor import services, audit
from factory_floor.config import COLLECTION_NAME, VECTOR_DIR
from factory_floor.vectorstore import get_embeddings, load_vectorstore

vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=get_embeddings())

req = services.DiagnosticRequest(
    question_text='F30021 ground fault after several hours of running — what should be checked?',
    machine_id='VFD-06', equipment_type='VFD',
    operator_id=operator.operator_id, tenant_id=operator.tenant_id, language='English',
)
result = services.run_diagnostic(req, vectorstore=vectorstore)
print('audit_id      :', result.audit_id)
print('safety.action :', (result.safety or {}).get('action'))
print('cost.usd      : $%.5f' % (result.cost or {}).get('total_usd', 0.0))

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


audit_id      : 1
safety.action : pass
cost.usd      : $0.00287


In [4]:
trail = audit.get_audit_trail(machine_id='VFD-06')
assert len(trail) == 1
row = trail[0]
assert row['id'] == result.audit_id
assert row['operator_id'] == 'OP-1001'
assert row['run_id'] == result.run_id
{k: row[k] for k in ('id', 'ts_utc', 'operator_id', 'machine_id', 'safety_action', 'cost_usd')}

{'id': 1,
 'ts_utc': '2026-08-27T21:53:12.427872+00:00',
 'operator_id': 'OP-1001',
 'machine_id': 'VFD-06',
 'safety_action': 'pass',
 'cost_usd': 0.0028712}

In [5]:
import sqlite3
with sqlite3.connect(get_settings().audit_db_path) as _c:
    _c.row_factory = sqlite3.Row
    srcs = [dict(r) for r in _c.execute('SELECT ordinal, source_file, page FROM recommendation_sources WHERE recommendation_id=?', (result.audit_id,))]
    tools = [dict(r) for r in _c.execute('SELECT tool, tool_input FROM tool_calls WHERE recommendation_id=?', (result.audit_id,))]
print('sources cited :', srcs)
print('tools called  :', tools)
assert srcs and tools

sources cited : [{'ordinal': 1, 'source_file': 'Siemens_G120_CU240BE2_List_Manual.pdf', 'page': 908}, {'ordinal': 2, 'source_file': 'Siemens_SINAMICS_G120C_List_Manual.pdf', 'page': 730}, {'ordinal': 3, 'source_file': 'Siemens_SINAMICS_G120C_List_Manual.pdf', 'page': 707}, {'ordinal': 4, 'source_file': 'Siemens_SINAMICS_G120C_List_Manual.pdf', 'page': 740}, {'ordinal': 5, 'source_file': 'Siemens_SINAMICS_G120C_List_Manual.pdf', 'page': 735}]
tools called  : [{'tool': 'search_manuals', 'tool_input': '{"query": "F30021 ground fault SINAMICS G120C"}'}]


## Operator records the resolution -> machine history

In [6]:
from factory_floor.machines import append_resolution_event, get_machine_history

before = get_machine_history('VFD-06')
ev_id = append_resolution_event(
    'VFD-06', operator_id=operator.operator_id,
    steps_text='Isolated the drive, measured insulation resistance motor-to-earth (0.3 MOhm), '
               'replaced the motor cable, retested. Fault cleared.',
    recommendation_id=result.audit_id,
)
after = get_machine_history('VFD-06', include_resolutions=True)
assert len(after) == len(before) + 1
live = [r for r in after if r['event_type'] == 'operator_resolution'][0]
assert live['technician'] == 'OP-1001'
print('new history row:', live)

new history row: {'event_id': 'R1', 'machine_id': 'VFD-06', 'event_date': '2026-08-27', 'event_type': 'operator_resolution', 'fault_code': '', 'description': 'Resolution recorded via the copilot by OP-1001', 'action_taken': 'Isolated the drive, measured insulation resistance motor-to-earth (0.3 MOhm), replaced the motor cable, retested. Fault cleared.', 'technician': 'OP-1001', 'downtime_hours': ''}


## CMMS / ERP export (demo — Opportunity #2)

A single button in the app. Here it appends the resolution record to a local outbox and stamps the event. A real integration replaces the file append with an authenticated POST to SAP PM / IBM Maximo or a webhook — the payload shape maps 1:1 to a work-order actuals record.

In [7]:
import json
ack = audit.export_to_cmms(ev_id)
print('ack:', ack)
assert ack['status'] == 'accepted'
outbox = get_settings().cmms_outbox_path
print('outbox line:', json.loads(Path(outbox).read_text().splitlines()[0]))
assert audit.get_resolution_events('VFD-06')[0]['cmms_exported_at'] is not None
print('resolution event stamped as exported — OK')

ack: {'status': 'accepted', 'cmms_ref': 'DEMO-000001', 'exported_at': '2026-08-27T21:53:12.454663+00:00'}
outbox line: {'exported_at': '2026-08-27T21:53:12.454663+00:00', 'resolution_event_id': 1, 'machine_id': 'VFD-06', 'operator_id': 'OP-1001', 'steps': 'Isolated the drive, measured insulation resistance motor-to-earth (0.3 MOhm), replaced the motor cable, retested. Fault cleared.'}
resolution event stamped as exported — OK
